In [ ]:
!pip install boto3

In [ ]:
import requests
import boto3
from botocore.client import Config
import json
import datetime
import locale
import re

minio_url = "http://minio:9000"
access_key = "minio"
secret_key = "minio123"

s3_client = boto3.client(
    's3',
    endpoint_url=minio_url,
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
    config=Config(signature_version='s3v4')
)

In [ ]:
def list_s3_directory(bucket_name, prefix=""):
    """
    Lists the contents of an S3 bucket at a specific prefix.

    Parameters:
    - bucket_name: The name of the S3 bucket.
    - prefix: The prefix or "directory" in the bucket to list.

    Returns:
    - A list of object keys in the specified bucket and prefix.
    """
    objects = []
    paginator = s3_client.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket_name, Prefix=prefix):
        if "Contents" in page:
            for obj in page["Contents"]:
                objects.append(obj["Key"])
    
    return objects

# Example usage:
bucket_name = "velib"
prefix = "bronze"
directory_contents = list_s3_directory(bucket_name, prefix)
print("Directory contents:", directory_contents)

In [ ]:
def list_unique_dates(bucket_name, prefix=""):
    """
    Lists unique date values (YYYY-MM-DD) from S3 keys in the specified bucket and prefix.
    """
    date_pattern = re.compile(r"\d{4}-\d{2}-\d{2}")
    unique_dates = set()

    paginator = s3_client.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket_name, Prefix=prefix):
        if "Contents" in page:
            for obj in page["Contents"]:
                # Extract date from the key using regex
                match = date_pattern.search(obj["Key"])
                if match:
                    unique_dates.add(match.group(0))

    return sorted(unique_dates)

# Example usage:
bucket_name = "velib"
prefix = "bronze"
unique_dates = list_unique_dates(bucket_name, prefix)
print("Unique dates:", unique_dates)

In [ ]:
!pip install trino

In [ ]:
import trino

host, port, user = 'trino-coordinator', 8080, 'trino'
conn = trino.dbapi.connect(host=host, port=port, user=user)
cur = conn.cursor()

catalog, schema, table = 'minio', '_tech_velib', 'job_success_silver'
schema_location = 's3a://velib/_tech'


# Define the row to upsert
part_day_value = '2023-10-25'
job_start_value = '2023-10-25 08:00:00'
job_end_value = '2023-10-25 12:00:00'

# # Define the upsert queries
# upsert_queries = [
#     # Delete any existing row with the same part_day
#     f"DELETE FROM {catalog}.{schema}.{table} WHERE part_day = '{part_day_value}'",
    
#     # Insert the new row
#     f"""
#     INSERT INTO {catalog}.{schema}.{table} (part_day, job_start, job_end)
#     VALUES ('{part_day_value}', TIMESTAMP '{job_start_value}', TIMESTAMP '{job_end_value}')
#     """
# ]


# List of queries to execute
queries = [
    f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema} WITH (location = '{schema_location}')",
    f"""
    CREATE TABLE IF NOT EXISTS {catalog}.{schema}.{table} (
        part_day VARCHAR,
        job_start TIMESTAMP,
        job_end TIMESTAMP
    ) WITH (
        format = 'ORC',
        transactional = true
    )
    """,
    f"SELECT * FROM {catalog}.{schema}.{table} LIMIT 5"
]

# Execute each query in the list
for query in queries:
    try:
        cur.execute(query)
        # Check if the query is a SELECT query to fetch results
        if query.startswith("SELECT"):
            results = cur.fetchall()
            for row in results:
                print(row)
        else:
            print(f"Executed: {query}")
    except Exception as e:
        print(f"Error executing query: {query}. Error: {e}")

# Close the cursor and connection
cur.close()
conn.close()

In [ ]:
import trino

host, port, user = 'trino-coordinator', 8080, 'trino'
conn = trino.dbapi.connect(host=host, port=port, user=user)

def get_part_day_values(conn):
    catalog, schema, table = 'minio', '_tech_velib', 'job_success_silver'  # Table parameters
    cur = conn.cursor()  # Create a cursor from the connection
    query = f"SELECT part_day FROM {catalog}.{schema}.{table}"  # Query to get part_day values
    try:
        cur.execute(query)  # Execute the query
        part_day_values = [row[0] for row in cur.fetchall()]  # Fetch all results in part_day column
        return part_day_values  # Return the list of part_day values
    except Exception as e:
        print(f"Error executing query: {query}. Error: {e}")  # Print error if query fails
        return []  # Return an empty list if there's an error
    finally:
        cur.close()  # Close the cursor
    
part_day_done = get_part_day_values(conn)
print("part_day_done=", part_day_done)

import requests
import boto3
from botocore.client import Config
import json
import datetime
import locale
import re

minio_url = "http://minio:9000"
access_key = "minio"
secret_key = "minio123"

s3_client = boto3.client(
    's3',
    endpoint_url=minio_url,
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
    config=Config(signature_version='s3v4')
)
def list_unique_dates(bucket_name, prefix=""):
    """
    Lists unique date values (YYYY-MM-DD) from S3 keys in the specified bucket and prefix.
    """
    date_pattern = re.compile(r"\d{4}-\d{2}-\d{2}")
    unique_dates = set()

    paginator = s3_client.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket_name, Prefix=prefix):
        if "Contents" in page:
            for obj in page["Contents"]:
                # Extract date from the key using regex
                match = date_pattern.search(obj["Key"])
                if match:
                    unique_dates.add(match.group(0))

    return sorted(unique_dates)

# Example usage:
bucket_name = "velib"
prefix = "bronze"
unique_dates = list_unique_dates(bucket_name, prefix)
# print("Unique dates:", unique_dates)

part_day_all = unique_dates
print("part_day_all=", part_day_all)

part_day_todo = [item for item in part_day_all if item not in part_day_done]
print("part_day_todo=", part_day_todo)

# Code final

In [21]:
import trino
import boto3
from botocore.client import Config
import re


def connect_to_trino(host='trino-coordinator', port=8080, user='trino'):
    """
    Creates and returns a Trino connection.
    """
    return trino.dbapi.connect(host=host, port=port, user=user)


def get_part_day_values(conn, catalog, schema, table):
    """
    Retrieves 'part_day' values from a specified table in Trino.
    """
    query = f"SELECT part_day FROM {catalog}.{schema}.{table}"  # Query to get part_day values
    cur = conn.cursor()
    try:
        cur.execute(query)  # Execute the query
        part_day_values = [row[0] for row in cur.fetchall()]  # Fetch all results in part_day column
        return part_day_values  # Return the list of part_day values
    except Exception as e:
        print(f"Error executing query: {query}. Error: {e}")  # Print error if query fails
        return []  # Return an empty list if there's an error
    finally:
        cur.close()  # Close the cursor


def create_s3_client(endpoint_url="http://minio:9000", access_key="minio", secret_key="minio123"):
    """
    Creates and returns an S3 client with the given parameters.
    """
    return boto3.client(
        's3',
        endpoint_url=endpoint_url,
        aws_access_key_id=access_key,
        aws_secret_access_key=secret_key,
        config=Config(signature_version='s3v4')
    )


def list_unique_dates(s3_client, bucket_name, prefix=""):
    """
    Lists unique date values (YYYY-MM-DD) from S3 keys in the specified bucket and prefix.
    """
    date_pattern = re.compile(r"\d{4}-\d{2}-\d{2}")
    unique_dates = set()
    paginator = s3_client.get_paginator("list_objects_v2")
    
    for page in paginator.paginate(Bucket=bucket_name, Prefix=prefix):
        for obj in page.get("Contents", []):
            match = date_pattern.search(obj["Key"])  # Assigning match without walrus operator
            if match:  # Checking if match is not None
                unique_dates.add(match.group(0))

    return sorted(unique_dates)


# List of queries to execute
def execute_queries(conn, catalog, schema, table, schema_location):
    """
    Executes a list of queries in Trino for setting up schema and table, and retrieves sample data.
    """
    queries = [
        f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema} WITH (location = '{schema_location}')",
        f"""
        CREATE TABLE IF NOT EXISTS {catalog}.{schema}.{table} (
            part_day VARCHAR,
            job_end TIMESTAMP
        ) WITH (
            format = 'ORC',
            transactional = true
        )
        """,
        f"SELECT * FROM {catalog}.{schema}.{table} LIMIT 5"
    ]
    
    cur = conn.cursor()
    for query in queries:
        try:
            cur.execute(query)
            # Check if the query is a SELECT query to fetch results
            if query.startswith("SELECT"):
                results = cur.fetchall()
                for row in results:
                    print(row)
            else:
                print(f"Executed: {query}")
        except Exception as e:
            raise Exception(f"Error executing query: {query}. Error: {e}")
    cur.close()

def build_part_day_todo(host, port, user, catalog, schema, table, schema_location):
    # Connect to Trino
    conn = connect_to_trino(host, port, user)

    # Execute the queries using the `execute_queries` function
    execute_queries(conn, catalog, schema, table, schema_location)

    # Retrieve part_day_done
    part_day_done = get_part_day_values(conn, catalog, schema, table)
    print("part_day_done =", part_day_done)

    # Create S3 client and retrieve unique dates
    s3_client = create_s3_client()
    bucket_name = "velib"
    prefix = "bronze"
    part_day_all = list_unique_dates(s3_client, bucket_name, prefix)
    print("part_day_all =", part_day_all)

    # Calculate part_day_todo
    part_day_todo = [item for item in part_day_all if item not in part_day_done]
    print("part_day_todo =", part_day_todo)

    conn.close()
    
    return part_day_todo

build_part_day_todo(
    host='trino-coordinator',
    port=8080,
    user='trino',
    catalog='minio',
    schema='_tech_velib',
    table='job_success_silver',
    schema_location='s3a://velib/_tech'
)

Executed: CREATE SCHEMA IF NOT EXISTS minio._tech_velib WITH (location = 's3a://velib/_tech')
Executed: 
        CREATE TABLE IF NOT EXISTS minio._tech_velib.job_success_silver (
            part_day VARCHAR,
            job_end TIMESTAMP
        ) WITH (
            format = 'ORC',
            transactional = true
        )
        
['2024-10-29', datetime.datetime(2024, 10, 31, 0, 8, 1, 510000)]
['2024-01-02', datetime.datetime(2024, 10, 31, 0, 7, 11, 919000)]
['2024-10-30', datetime.datetime(2024, 10, 31, 0, 12, 25, 128000)]
part_day_done = ['2024-10-30', '2024-10-29', '2024-01-02']
part_day_all = ['2024-10-29', '2024-10-30', '2024-10-31']
part_day_todo = ['2024-10-31']


['2024-10-31']

In [3]:
import trino

def upsert_job_success(**kwargs):
    part_day_value = kwargs['part_day_value']
    host = kwargs['host']
    port = kwargs['port']
    user = kwargs['user']
    catalog = kwargs['catalog']
    schema = kwargs['schema']
    table_name = kwargs['table_name']

    conn = trino.dbapi.connect(host=host, port=port, user=user)
    merge_query = f"""
    MERGE INTO {catalog}.{schema}.{table_name} t
    USING (
        VALUES ('{part_day_value}', CURRENT_TIMESTAMP)
    ) AS s (part_day, job_end)
    ON t.part_day = s.part_day
    WHEN MATCHED THEN UPDATE SET job_end = s.job_end
    WHEN NOT MATCHED THEN INSERT (part_day, job_end) VALUES (s.part_day, CURRENT_TIMESTAMP)
    """
    cur = conn.cursor()
    try:
        # Execute the MERGE query
        cur.execute(merge_query)
        print(f"Executed: {merge_query}")
        print("Merge query executed successfully.")
    except Exception as e:
        raise Exception(f"Error executing merge query: {e}")
    finally:
        cur.close()
        conn.close()
        
op_kwargs={
            'part_day_value': '2024-01-28',
            'host': 'trino-coordinator',
            'port': '8080',
            'user': 'trino',
            'catalog': 'minio',
            'schema': '_tech_velib',
            'table_name': 'job_success_silver'
        }

upsert_job_success(**op_kwargs)

Executed: 
    MERGE INTO minio._tech_velib.job_success_silver t
    USING (
        VALUES ('2024-01-28', CURRENT_TIMESTAMP)
    ) AS s (part_day, job_end)
    ON t.part_day = s.part_day
    WHEN MATCHED THEN UPDATE SET job_end = s.job_end
    WHEN NOT MATCHED THEN INSERT (part_day, job_end) VALUES (s.part_day, CURRENT_TIMESTAMP)
    
Merge query executed successfully.


In [23]:
import trino
import time

def row_exists(connection, **kwargs):
    part_day_value = kwargs['part_day_value']
    host = kwargs['host']
    port = kwargs['port']
    user = kwargs['user']
    catalog = kwargs['catalog']
    schema = kwargs['schema']
    table_name = kwargs['table_name']
    
    select_query = f"""
    SELECT 1 FROM {catalog}.{schema}.{table_name}
    WHERE part_day = '{part_day_value}'
    LIMIT 1
    """
    conn = connection
    cur = conn.cursor()
    try:
        # Check if the row exists
        cur.execute(select_query)
        print(f"Executed: {select_query}")
        result = cur.fetchone()
        return result is not None
    except Exception as e:
        raise Exception(f"Error checking row existence: {e}")
    finally:
        cur.close()
        conn.close()

def upsert_job_success(connection, **kwargs):
    part_day_value = kwargs['part_day_value']
    host = kwargs['host']
    port = kwargs['port']
    user = kwargs['user']
    catalog = kwargs['catalog']
    schema = kwargs['schema']
    table_name = kwargs['table_name']
    
    conn = connection
    merge_query = f"""
    MERGE INTO {catalog}.{schema}.{table_name} t
    USING (
        VALUES ('{part_day_value}', CURRENT_TIMESTAMP)
    ) AS s (part_day, job_end)
    ON t.part_day = s.part_day
    WHEN MATCHED THEN UPDATE SET job_end = s.job_end
    WHEN NOT MATCHED THEN INSERT (part_day, job_end) VALUES (s.part_day, CURRENT_TIMESTAMP)
    """
    cur = conn.cursor()
    try:
        # Execute the MERGE query
        cur.execute(merge_query)
        print(f"Executed: {merge_query}")
        print("Merge query executed successfully.")
    except Exception as e:
        raise Exception(f"Error executing merge query: {e}")
    finally:
        cur.close()
        conn.close()

def upsert_job_success_with_retries(**kwargs):
    part_day_value = kwargs['part_day_value']
    host = kwargs['host']
    port = kwargs['port']
    user = kwargs['user']
    catalog = kwargs['catalog']
    schema = kwargs['schema']
    table_name = kwargs['table_name']
    
    max_retries = kwargs['max_retries']
    delay = kwargs['delay']
    
    conn = trino.dbapi.connect(host=host, port=port, user=user)
    
    attempt = 0
    while attempt < max_retries:
        try:
            upsert_job_success(conn, **kwargs)
            # Check if the row was successfully inserted
            if row_exists(conn, **kwargs):
                print("Row exists in the table.")
                break  # Row is confirmed, exit the loop
            else:
                print("Row not found after upsert. Retrying...")
        except Exception as e:
            print(f"Attempt {attempt + 1} failed with error: {e}")
        
        attempt += 1
        if attempt < max_retries:
            print(f"Retrying in {delay} seconds...")
            time.sleep(delay)
        else:
            print("Max retries reached. Row not inserted.")
            raise Exception("Failed to confirm row insertion after multiple retries.")

op_kwargs = {
    'part_day_value': '2024-01-28',
    'host': 'trino-coordinator',
    'port': '8080',
    'user': 'trino',
    'catalog': 'minio',
    'schema': '_tech_velib',
    'table_name': 'job_success_silver',
    'max_retries': 5,
    'delay': 2
}

# import logging
# logging.basicConfig()
# logging.getLogger().setLevel(logging.DEBUG)

# Call the function with retries
upsert_with_retries(**op_kwargs)

DEBUG:urllib3.connectionpool:Starting new HTTP connection (1): trino-coordinator:8080
DEBUG:urllib3.connectionpool:http://trino-coordinator:8080 "POST /v1/statement HTTP/1.1" 200 319
DEBUG:urllib3.connectionpool:http://trino-coordinator:8080 "GET /v1/statement/queued/20241031_095212_03943_xf3mb/y7b10e5ea49f778832c921a136438edbe4f1bbfc6/1 HTTP/1.1" 200 318
DEBUG:urllib3.connectionpool:http://trino-coordinator:8080 "GET /v1/statement/queued/20241031_095212_03943_xf3mb/y9ee55add517de63ca1ae1afe026f72ad0c750f15/2 HTTP/1.1" 200 328
DEBUG:urllib3.connectionpool:http://trino-coordinator:8080 "GET /v1/statement/executing/20241031_095212_03943_xf3mb/y301267d9b17d3494f96e87171952783f0d3decec/0 HTTP/1.1" 200 519
DEBUG:urllib3.connectionpool:http://trino-coordinator:8080 "GET /v1/statement/executing/20241031_095212_03943_xf3mb/ydc69c86d04689aedd6b8c384958f791b9411c3a8/1 HTTP/1.1" 200 686
DEBUG:urllib3.connectionpool:http://trino-coordinator:8080 "DELETE /v1/statement/executing/20241031_095212_0394

Executed: 
    MERGE INTO minio._tech_velib.job_success_silver t
    USING (
        VALUES ('2024-01-28', CURRENT_TIMESTAMP)
    ) AS s (part_day, job_end)
    ON t.part_day = s.part_day
    WHEN MATCHED THEN UPDATE SET job_end = s.job_end
    WHEN NOT MATCHED THEN INSERT (part_day, job_end) VALUES (s.part_day, CURRENT_TIMESTAMP)
    
Merge query executed successfully.


DEBUG:urllib3.connectionpool:http://trino-coordinator:8080 "GET /v1/statement/executing/20241031_095213_03944_xf3mb/y26a9680f4317779bf17b8073a91efad64ae17bc7/1 HTTP/1.1" 200 488
DEBUG:urllib3.connectionpool:http://trino-coordinator:8080 "GET /v1/statement/executing/20241031_095213_03944_xf3mb/yb0de7c89b84be33f03dac95d65a06eb7e3aa0ec2/2 HTTP/1.1" 200 503
DEBUG:urllib3.connectionpool:http://trino-coordinator:8080 "GET /v1/statement/executing/20241031_095213_03944_xf3mb/y53bd0e2608c1ae9b306dc5caeefd4a931015350b/3 HTTP/1.1" 200 498
DEBUG:urllib3.connectionpool:http://trino-coordinator:8080 "GET /v1/statement/executing/20241031_095213_03944_xf3mb/ycda4b13c76832b123a4cbfc63e0684ba60411e4e/4 HTTP/1.1" 200 499
DEBUG:urllib3.connectionpool:http://trino-coordinator:8080 "GET /v1/statement/executing/20241031_095213_03944_xf3mb/y4c3f97b5c104776124dbd26a626b22c772f2f57a/5 HTTP/1.1" 200 498
DEBUG:urllib3.connectionpool:http://trino-coordinator:8080 "GET /v1/statement/executing/20241031_095213_03944_

Executed: 
    SELECT 1 FROM minio._tech_velib.job_success_silver
    WHERE part_day = '2024-01-28'
    LIMIT 1
    
Row exists in the table.


In [1]:
import pkg_resources
trino_version = pkg_resources.get_distribution('trino').version
print(f"trino_version = {trino_version}")


trino_version = 0.330.0


In [4]:
import trino
import time

def row_exists(connection, **kwargs):
    part_day_value = kwargs['part_day_value']
    host = kwargs['host']
    port = kwargs['port']
    user = kwargs['user']
    catalog = kwargs['catalog']
    schema = kwargs['schema']
    table_name = kwargs['table_name']
    
    select_query = f"""
    SELECT 1 FROM {catalog}.{schema}.{table_name}
    WHERE part_day = '{part_day_value}'
    LIMIT 1
    """
    conn = connection
    cur = conn.cursor()
    try:
        # Check if the row exists
        cur.execute(select_query)
        print(f"Executed: {select_query}")
        result = cur.fetchone()
        return result is not None
    except Exception as e:
        raise Exception(f"Error checking row existence: {e}")
    finally:
        cur.close()
        conn.close()


def upsert_job_success(connection, **kwargs):
    part_day_value = kwargs['part_day_value']
    catalog = kwargs['catalog']
    schema = kwargs['schema']
    table_name = kwargs['table_name']
    
    conn = connection
    cur = conn.cursor()
    
    # Define the UPDATE query to set job_end for existing records
    update_query = f"""
    UPDATE {catalog}.{schema}.{table_name}
    SET job_end = CURRENT_TIMESTAMP
    WHERE part_day = '{part_day_value}'
    """
    
    # Define the INSERT query to add a new record if it doesn't exist
    insert_query = f"""
    INSERT INTO {catalog}.{schema}.{table_name} (part_day, job_end)
    SELECT '{part_day_value}', CURRENT_TIMESTAMP
    WHERE NOT EXISTS (
        SELECT 1 FROM {catalog}.{schema}.{table_name} WHERE part_day = '{part_day_value}'
    )
    """
    
    try:
        # Execute the UPDATE query
        cur.execute(update_query)
        print(f"Executed: {update_query}")
        
        # Execute the INSERT query
        cur.execute(insert_query)
        print(f"Executed: {insert_query}")
        
        print("Upsert operation completed successfully.")
    except Exception as e:
        raise Exception(f"Error executing upsert operation: {e}")
    finally:
        # Ensure the cursor and connection are closed
        cur.close()
        conn.close()

def upsert_job_success_with_retries(**kwargs):
    part_day_value = kwargs['part_day_value']
    host = kwargs['host']
    port = kwargs['port']
    user = kwargs['user']
    catalog = kwargs['catalog']
    schema = kwargs['schema']
    table_name = kwargs['table_name']
    
    max_retries = kwargs['max_retries']
    delay = kwargs['delay']
    
    conn = trino.dbapi.connect(host=host, port=port, user=user)
    
    attempt = 0
    while attempt < max_retries:
        try:
            upsert_job_success(conn, **kwargs)
            # Check if the row was successfully inserted
            if row_exists(conn, **kwargs):
                print("Row exists in the table.")
                break  # Row is confirmed, exit the loop
            else:
                print("Row not found after upsert. Retrying...")
        except Exception as e:
            print(f"Attempt {attempt + 1} failed with error: {e}")
        
        attempt += 1
        if attempt < max_retries:
            print(f"Retrying in {delay} seconds...")
            time.sleep(delay)
        else:
            print("Max retries reached. Row not inserted.")
            raise Exception("Failed to confirm row insertion after multiple retries.")

op_kwargs = {
    'part_day_value': '2024-01-15',
    'host': 'trino-coordinator',
    'port': '8080',
    'user': 'trino',
    'catalog': 'minio',
    'schema': '_tech_velib',
    'table_name': 'job_success_silver',
    'max_retries': 5,
    'delay': 2
}

# Call the function with retries
upsert_job_success_with_retries(**op_kwargs)

Executed: 
    UPDATE minio._tech_velib.job_success_silver
    SET job_end = CURRENT_TIMESTAMP
    WHERE part_day = '2024-01-15'
    
Executed: 
    INSERT INTO minio._tech_velib.job_success_silver (part_day, job_end)
    SELECT '2024-01-15', CURRENT_TIMESTAMP
    WHERE NOT EXISTS (
        SELECT 1 FROM minio._tech_velib.job_success_silver WHERE part_day = '2024-01-15'
    )
    
Upsert operation completed successfully.
Executed: 
    SELECT 1 FROM minio._tech_velib.job_success_silver
    WHERE part_day = '2024-01-15'
    LIMIT 1
    
Row not found after upsert. Retrying...
Retrying in 2 seconds...
Executed: 
    UPDATE minio._tech_velib.job_success_silver
    SET job_end = CURRENT_TIMESTAMP
    WHERE part_day = '2024-01-15'
    
Executed: 
    INSERT INTO minio._tech_velib.job_success_silver (part_day, job_end)
    SELECT '2024-01-15', CURRENT_TIMESTAMP
    WHERE NOT EXISTS (
        SELECT 1 FROM minio._tech_velib.job_success_silver WHERE part_day = '2024-01-15'
    )
    
Upsert op